# AU vs AI — Drawing Model Training

Trains the Round 2 classifier on Google's QuickDraw dataset and exports it for
TensorFlow.js.

**Runtime: about 20-30 minutes.** `Runtime -> Change runtime type -> T4 GPU`
makes it roughly 4x faster and is free.

Run every cell top to bottom. The last cell downloads a zip you drop into the
project.

---

### Two deliberate choices

**Keras 2, not Keras 3.** Colab ships Keras 3, whose saved models are not
loadable by TensorFlow.js. The first cell installs `tf_keras` (Google's
maintained Keras 2) and switches to it before TensorFlow is imported.

**No `tensorflowjs` package.** On Colab's Python 3.13 there is no working wheel,
and pip silently falls back to a years-old version that crashes on import. This
notebook writes the TensorFlow.js format directly instead — about forty lines,
no dependency, and it cannot break when a package updates.

The output format has been verified against the actual TensorFlow.js runtime.

In [ ]:
!pip install -q tf_keras

import os

# MUST be set before TensorFlow is imported anywhere in this runtime.
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf
import tf_keras as keras
from tf_keras import layers

print('TensorFlow', tf.__version__)
print('Keras     ', keras.__version__, '(expect 2.x)')
print('GPU       ', tf.config.list_physical_devices('GPU') or 'none (CPU is fine, just slower)')

assert keras.__version__.startswith('2'), (
    'Keras 2 required. Runtime -> Restart session, then run this cell first.'
)

## 1. Download the data

QuickDraw's `numpy_bitmap` files are already 28x28, centred and cropped — the
exact format the game's canvas produces after preprocessing. Using the bitmap
files rather than raw stroke data keeps training simple and the browser model
small.

In [ ]:
import urllib.request
import numpy as np

CLASSES = [
    'bicycle', 'cat', 'fish', 'car', 'tree', 'cup', 'star', 'umbrella',
    'clock', 'airplane', 'apple', 'house', 'key', 'ladder', 'sun',
]

# Samples per class. 20k is plenty and keeps training quick.
PER_CLASS = 20000

BASE = 'https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap'
os.makedirs('data', exist_ok=True)

for name in CLASSES:
    path = f'data/{name}.npy'
    if os.path.exists(path):
        print(f'  {name:10s} cached')
        continue
    url = f'{BASE}/{name.replace("_", "%20")}.npy'
    urllib.request.urlretrieve(url, path)
    print(f'  {name:10s} downloaded')

print()
print('All classes ready.')

In [ ]:
X_parts, y_parts = [], []

for idx, name in enumerate(CLASSES):
    arr = np.load(f'data/{name}.npy')          # (N, 784) uint8
    take = min(PER_CLASS, len(arr))
    # Shuffle before slicing: the files are not randomly ordered.
    rng = np.random.default_rng(42 + idx)
    sel = rng.choice(len(arr), size=take, replace=False)
    X_parts.append(arr[sel])
    y_parts.append(np.full(take, idx, dtype=np.int64))
    print(f'  {name:10s} {take:6d} samples')

X = np.concatenate(X_parts).astype('float32') / 255.0
y = np.concatenate(y_parts)
X = X.reshape(-1, 28, 28, 1)

perm = np.random.default_rng(7).permutation(len(X))
X, y = X[perm], y[perm]

split = int(len(X) * 0.9)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

print()
print(f'Train {X_train.shape}   Val {X_val.shape}')

## 2. Look at the data first

Never train without seeing what you are training on. These should look like
recognisable doodles, white on black — the same kind of picture the game's
`/debug/draw` page shows after preprocessing.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 5, figsize=(11, 7))
for ax, idx in zip(axes.flat, range(len(CLASSES))):
    sample = X_train[y_train == idx][0].reshape(28, 28)
    ax.imshow(sample, cmap='gray')
    ax.set_title(CLASSES[idx], fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Train

A deliberately small CNN, roughly 500KB exported. Every student downloads this
over venue wifi, so a bigger model would score slightly better on paper and be
worse at the actual event.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling2D(),

    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(CLASSES), activation='softmax'),
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        patience=3, restore_best_weights=True, monitor='val_accuracy'),
    keras.callbacks.ReduceLROnPlateau(
        factor=0.5, patience=2, monitor='val_loss'),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=256,
    callbacks=callbacks,
    verbose=1,
)

## 4. Per-class accuracy — read this carefully

Overall accuracy is the least useful number here. What matters is the **worst**
class, because a student assigned that word has a bad experience through no
fault of their own.

**Any class below about 80% should be replaced.** Edit `CLASSES`, re-run from
the download cell, retrain. That is a normal part of this process.

In [ ]:
probs = model.predict(X_val, batch_size=512, verbose=0)
preds = probs.argmax(axis=1)

print(f'Overall validation accuracy: {(preds == y_val).mean():.1%}')
print()
print(f'{"class":<12}{"accuracy":>10}      verdict')
print('-' * 46)

rows = [((preds[y_val == i] == i).mean(), n) for i, n in enumerate(CLASSES)]

for acc, name in sorted(rows):
    if acc >= 0.90:
        verdict = 'excellent'
    elif acc >= 0.80:
        verdict = 'fine'
    else:
        verdict = 'REPLACE THIS CLASS'
    print(f'{name:<12}{acc:>9.1%}      {verdict}')

## 5. Confusion check

Pairs that confuse the model will also confuse students. If two classes are
constantly swapped, drop one of them.

In [ ]:
from collections import Counter

mistakes = Counter()
for true_idx, pred_idx in zip(y_val, preds):
    if true_idx != pred_idx:
        mistakes[(CLASSES[true_idx], CLASSES[pred_idx])] += 1

print('Most common mistakes:')
print()
for (truth, guess), count in mistakes.most_common(10):
    print(f'  drew {truth:<10} -> model said {guess:<10} {count:>5} times')

## 6. Export for the browser

Writes the TensorFlow.js `LayersModel` format directly: a `model.json` holding
the Keras config plus a weights manifest, and one binary shard containing the
weight values concatenated in manifest order.

Weight names follow the convention TensorFlow.js matches against —
`<layer_name>/<weight>`, e.g. `conv2d/kernel`. Getting this wrong is what
produces *"Provided weight data has no target variable"*, so the next cell
verifies it.

In [ ]:
import json
import numpy as np

OUT = 'quickdraw'
os.makedirs(OUT, exist_ok=True)

# ---------------------------------------------------------------------------
# Collect weights in the order TensorFlow.js expects, naming each one
# "<layer>/<weight>" — the same convention the official converter uses.
# ---------------------------------------------------------------------------
specs = []
chunks = []

for layer in model.layers:
    for weight in layer.weights:
        # Keras variable names look like 'conv2d/kernel:0'; tfjs wants the
        # ':0' suffix removed.
        name = weight.name.split(':')[0]
        if '/' not in name:
            name = f'{layer.name}/{name}'

        value = np.asarray(weight.numpy(), dtype=np.float32)
        specs.append({
            'name': name,
            'shape': list(value.shape),
            'dtype': 'float32',
        })
        chunks.append(value.ravel())

blob = np.concatenate(chunks).astype('<f4')   # little-endian float32
SHARD = 'group1-shard1of1.bin'
blob.tofile(os.path.join(OUT, SHARD))

# ---------------------------------------------------------------------------
# model.json
# ---------------------------------------------------------------------------
spec = {
    'format': 'layers-model',
    'generatedBy': f'keras v{keras.__version__}',
    'convertedBy': 'AU vs AI notebook',
    'modelTopology': {
        'keras_version': keras.__version__,
        'backend': 'tensorflow',
        'model_config': json.loads(model.to_json()),
    },
    'weightsManifest': [{'paths': [SHARD], 'weights': specs}],
}

with open(f'{OUT}/model.json', 'w') as f:
    json.dump(spec, f)

# Class order MUST match the model's output order. Reading labels from disk
# means a retrain with different classes cannot silently mislabel everything.
with open(f'{OUT}/labels.json', 'w') as f:
    json.dump(CLASSES, f, indent=2)

total = sum(os.path.getsize(os.path.join(OUT, f)) for f in os.listdir(OUT))
print(f'{len(specs)} weight tensors, {blob.size:,} floats')
print(f'Exported {len(os.listdir(OUT))} files, {total / 1024:.0f} KB total')
print('Files:', sorted(os.listdir(OUT)))

## 7. Verify the export

Re-reads the files from disk and checks everything TensorFlow.js will check:
that the config is Keras 2, that every layer weight appears in the manifest, and
that the binary is exactly the size the manifest claims.

**If this prints FAIL, do not download.** Fix it here rather than discovering it
in the browser.

In [ ]:
with open(f'{OUT}/model.json') as f:
    check = json.load(f)

problems = []

# 1. Keras 2 config
kv = check['modelTopology'].get('keras_version', '')
if not kv.startswith('2'):
    problems.append(f'Config is Keras {kv}; TensorFlow.js needs Keras 2.')

topology = json.dumps(check['modelTopology'])
if '"batch_shape"' in topology:
    problems.append('Found batch_shape (Keras 3 naming).')
if '"DTypePolicy"' in topology:
    problems.append('Found DTypePolicy dtype objects (Keras 3 naming).')

# 2. Every weight the model has must be in the manifest
manifest = check['weightsManifest'][0]['weights']
manifest_names = {w['name'] for w in manifest}
expected = set()
for layer in model.layers:
    for weight in layer.weights:
        n = weight.name.split(':')[0]
        expected.add(n if '/' in n else f'{layer.name}/{n}')

missing = expected - manifest_names
if missing:
    problems.append(f'Weights missing from manifest: {sorted(missing)[:5]}')

# 3. Binary size must match the manifest exactly
declared = sum(int(np.prod(w['shape'])) for w in manifest) * 4
actual = os.path.getsize(os.path.join(OUT, check['weightsManifest'][0]['paths'][0]))
if declared != actual:
    problems.append(f'Shard is {actual} bytes, manifest declares {declared}.')

# 4. Output width must match the label count
if model.output_shape[-1] != len(CLASSES):
    problems.append(
        f'Model outputs {model.output_shape[-1]} classes, labels.json has {len(CLASSES)}.'
    )

print(f'keras_version:  {kv}')
print(f'weight tensors: {len(manifest)}')
print(f'shard bytes:    {actual:,}')
print(f'output classes: {model.output_shape[-1]}')
print()

if problems:
    print('FAIL')
    for item in problems:
        print(f'  - {item}')
else:
    print('PASS — this export will load in TensorFlow.js.')

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('quickdraw-model', 'zip', OUT)
files.download('quickdraw-model.zip')

## 8. Install it

1. **Delete everything currently in `public/models/quickdraw/`** except
   `README.md` — including any `model.original.json` left by the old patch
   script
2. Unzip `quickdraw-model.zip` into that folder
3. You should have exactly `model.json`, one or more `.bin` files, and
   `labels.json`
4. Restart the dev server, hard-refresh **`/debug/draw`**

It should read **`Classifier: tfjs — real model loaded`**. Draw a key, a star, a
house — each should come out well ahead rather than everything tying.